# SnapUGC-LightKD Clean-Metadata 5000-Video Run

This notebook tests the hypothesis that training/evaluating on a metadata-clean subset improves ECR prediction.

It filters the original training CSV before subset selection, then runs the same LightKD pipeline:

- drop rows with missing metadata according to `REQUIRE_TEXT`
- sample a deterministic ECR-stratified 5000-video subset from the clean pool
- DOVER-Mobile quality scoring
- CLIP/R(2+1)D/YAMNet/Sentence-T5/BLIP-base feature extraction
- Teacher, Student baseline, and Student+KD training

Default setting: `REQUIRE_TEXT = "complete"`, meaning both `Title` and `Description` must be present. Change to `"any"` if you want to keep rows that have at least one text field.


In [ ]:
# 0. Clone this repository into Kaggle working directory
GITHUB_REPO = "https://github.com/TranTop2806/SnapUGC-LightKD.git"
REPO_DIR = "/kaggle/working/SnapUGC-LightKD"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
    %cd /kaggle/working

print("Repo ready:", REPO_DIR)


In [ ]:
# 1. Install dependencies
!pip install -q -U "transformers>=4.49.0" accelerate sentence-transformers
!pip install -q eva-decord tensorflow tensorflow_hub scipy pandas tqdm matplotlib

import os, sys, glob, json, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2. Paths and clean-metadata ECR-balanced 5000-video subset
MAX_VIDEOS = 5000
SUBSET_SEED = 42
ECR_BINS = 10
REQUIRE_TEXT = 'complete'  # 'complete' = keep rows with both Title+Description; 'any' = keep rows with at least one
RUN_DOVER = True
RUN_CAPTION = True
RESET_FEATURES = True
CAPTION_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

from pathlib import Path

def count_video_files(path):
    if not path or not path.is_dir():
        return 0
    exts = {'.mp4', '.mov', '.mkv', '.webm', '.avi'}
    return sum(1 for item in path.iterdir() if item.is_file() and item.suffix.lower() in exts)

input_candidates = []
def add_input_candidate(path):
    path = Path(path)
    if path.exists() and (path / 'train_data.csv').exists() and path not in input_candidates:
        input_candidates.append(path)

add_input_candidate('/kaggle/input/datasets/nguyntuncng/snapugc-dataset')
for train_csv in Path('/kaggle/input').rglob('train_data.csv'):
    add_input_candidate(train_csv.parent)

if not input_candidates:
    raise FileNotFoundError('Cannot find any Kaggle input directory containing train_data.csv')

selected = None
for candidate in input_candidates:
    video_candidates = [
        candidate / 'train_videos' / 'train_videos',
        candidate / 'train_videos',
    ]
    video_candidates += [p for p in candidate.glob('**/train_videos') if p.is_dir()]
    video_candidates = list(dict.fromkeys(video_candidates))
    best_video_dir = max(video_candidates, key=count_video_files, default=None)
    if best_video_dir and count_video_files(best_video_dir) > 0:
        selected = (candidate, best_video_dir)
        break

if selected is None:
    details = {str(c): [str(p) for p in [c / 'train_videos' / 'train_videos', c / 'train_videos'] if p.exists()] for c in input_candidates}
    raise FileNotFoundError(f'Found train_data.csv but no train video directory with files. Candidates: {details}')

INPUT_DIR = str(selected[0])
TRAIN_VIDEO_ROOT = str(selected[1])
TRAIN_CSV_ORIG = os.path.join(INPUT_DIR, 'train_data.csv')

REPO_CANDIDATES = [
    globals().get('REPO_DIR', '/kaggle/working/SnapUGC-LightKD'),
    '/kaggle/working/SnapUGC-LightKD',
]
REPO_DIR = next((p for p in REPO_CANDIDATES if p and os.path.exists(os.path.join(p, 'scripts'))), None)
if REPO_DIR is None:
    raise FileNotFoundError('Cannot find SnapUGC-LightKD. Check the clone cell or update REPO_DIR.')

OUTPUT_DIR = f'/kaggle/working/clean_{REQUIRE_TEXT}_metadata_{MAX_VIDEOS}_videos'
SUBSET_VIDEO_DIR = os.path.join(OUTPUT_DIR, 'subset_videos')
CLEAN_SOURCE_CSV = os.path.join(OUTPUT_DIR, f'train_clean_{REQUIRE_TEXT}_metadata.csv')
SUBSET_CSV = os.path.join(OUTPUT_DIR, f'train_subset_clean_{REQUIRE_TEXT}_{MAX_VIDEOS}.csv')
FEATURES_PATH = os.path.join(OUTPUT_DIR, f'features_clean_{REQUIRE_TEXT}_{MAX_VIDEOS}.json')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
DOVER_DIR = os.path.join(OUTPUT_DIR, 'dover')
DOVER_CSV = os.path.join(DOVER_DIR, 'dover_scores.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOVER_DIR, exist_ok=True)

import pandas as pd, subprocess

def missing_text_mask(series):
    text = series.astype('string').fillna('').str.strip()
    return text.eq('') | text.str.lower().isin(['nan', 'none', 'null', 'na', 'n/a'])

raw_df = pd.read_csv(TRAIN_CSV_ORIG)
required_cols = ['Id', 'Title', 'Description', 'ECR']
missing_cols = [c for c in required_cols if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f'Missing expected columns: {missing_cols}')

title_missing = missing_text_mask(raw_df['Title'])
description_missing = missing_text_mask(raw_df['Description'])
both_missing = title_missing & description_missing
either_missing = title_missing | description_missing

if REQUIRE_TEXT == 'complete':
    clean_df = raw_df.loc[~either_missing].copy()
elif REQUIRE_TEXT == 'any':
    clean_df = raw_df.loc[~both_missing].copy()
else:
    raise ValueError("REQUIRE_TEXT must be 'complete' or 'any'")

clean_df['Title'] = clean_df['Title'].fillna('').astype(str).str.strip()
clean_df['Description'] = clean_df['Description'].fillna('').astype(str).str.strip()
clean_df.to_csv(CLEAN_SOURCE_CSV, index=False)

def ecr_stats(df):
    ecr = pd.to_numeric(df['ECR'], errors='coerce')
    return {
        'rows': int(len(df)),
        'min': float(ecr.min()),
        'max': float(ecr.max()),
        'mean': float(ecr.mean()),
        'std': float(ecr.std(ddof=0)),
        'p25': float(ecr.quantile(0.25)),
        'p50': float(ecr.quantile(0.50)),
        'p75': float(ecr.quantile(0.75)),
    }

print('Raw rows:', len(raw_df))
print('Raw missing Title:', int(title_missing.sum()), f'{title_missing.mean()*100:.2f}%')
print('Raw missing Description:', int(description_missing.sum()), f'{description_missing.mean()*100:.2f}%')
print('Raw missing both:', int(both_missing.sum()), f'{both_missing.mean()*100:.2f}%')
print('Raw missing either:', int(either_missing.sum()), f'{either_missing.mean()*100:.2f}%')
print('Raw ECR stats:', ecr_stats(raw_df))
print('Clean mode:', REQUIRE_TEXT)
print('Clean source rows:', len(clean_df))
print('Clean source ECR stats:', ecr_stats(clean_df))
print('CLEAN_SOURCE_CSV:', CLEAN_SOURCE_CSV)
if len(clean_df) < MAX_VIDEOS:
    raise RuntimeError(f'Only {len(clean_df)} clean rows available, cannot sample {MAX_VIDEOS}.')

subset_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/make_subset.py'),
    '--csv', CLEAN_SOURCE_CSV,
    '--videos', TRAIN_VIDEO_ROOT,
    '--out-csv', SUBSET_CSV,
    '--out-videos', SUBSET_VIDEO_DIR,
    '--max', str(MAX_VIDEOS),
    '--seed', str(SUBSET_SEED),
    '--bins', str(ECR_BINS),
]
if RESET_FEATURES:
    subset_cmd.append('--reset')
print(' '.join(subset_cmd))
subprocess.run(subset_cmd, check=True)

TRAIN_CSV = SUBSET_CSV
TRAIN_VIDEO_DIR = SUBSET_VIDEO_DIR
if RESET_FEATURES:
    for stale_path in [FEATURES_PATH, DOVER_CSV]:
        if os.path.exists(stale_path):
            os.remove(stale_path)

subset_df = pd.read_csv(TRAIN_CSV)
subset_title_missing = missing_text_mask(subset_df['Title'])
subset_description_missing = missing_text_mask(subset_df['Description'])
subset_both_missing = subset_title_missing & subset_description_missing
subset_either_missing = subset_title_missing | subset_description_missing
ecr = pd.to_numeric(subset_df['ECR'], errors='coerce')
print('INPUT_DIR:', INPUT_DIR)
print('TRAIN_CSV:', TRAIN_CSV)
print('TRAIN_VIDEO_DIR:', TRAIN_VIDEO_DIR)
print('Subset rows:', len(subset_df))
print('Subset videos:', len(glob.glob(os.path.join(TRAIN_VIDEO_DIR, '*.mp4'))))
print('Subset missing Title:', int(subset_title_missing.sum()), f'{subset_title_missing.mean()*100:.2f}%')
print('Subset missing Description:', int(subset_description_missing.sum()), f'{subset_description_missing.mean()*100:.2f}%')
print('Subset missing both:', int(subset_both_missing.sum()), f'{subset_both_missing.mean()*100:.2f}%')
print('Subset missing either:', int(subset_either_missing.sum()), f'{subset_either_missing.mean()*100:.2f}%')
print('Subset ECR:', ecr_stats(subset_df))
print('Subset ECR quantile counts:')
print(pd.qcut(ecr, q=min(ECR_BINS, ecr.nunique(), len(ecr)), duplicates='drop').value_counts().sort_index())
print('REPO_DIR:', REPO_DIR)
print('FEATURES_PATH:', FEATURES_PATH)
print('CAPTION_DEVICE:', CAPTION_DEVICE)


In [ ]:
# 3. DOVER-Mobile quality scores for the bounded subset
# DOVER-Mobile is the official lightweight DOVER variant and is used for the full 5000-video subset.
if RUN_DOVER:
    !git clone -q https://github.com/VQAssessment/DOVER.git /kaggle/working/DOVER 2>/dev/null || true
    %cd /kaggle/working/DOVER
    !pip install -q scikit-video yacs timm einops opencv-python-headless decord pyyaml pandas scipy tqdm
    !mkdir -p pretrained_weights
    !wget -q -nc https://github.com/QualityAssessment/DOVER/releases/download/v0.5.0/DOVER-Mobile.pth -O pretrained_weights/DOVER-Mobile.pth
    !ls -lh pretrained_weights
    !python evaluate_a_set_of_videos.py -in "$TRAIN_VIDEO_DIR" -out "$DOVER_CSV" -o dover-mobile.yml
    print('DOVER_CSV:', DOVER_CSV)
    if not os.path.exists(DOVER_CSV):
        raise RuntimeError(f'DOVER did not produce expected CSV: {DOVER_CSV}')
    import pandas as pd
    dover_preview = pd.read_csv(DOVER_CSV)
    print('DOVER rows:', len(dover_preview))
    print(dover_preview.head())
    if len(dover_preview) < max(1, int(MAX_VIDEOS * 0.8)):
        raise RuntimeError(f'DOVER produced too few rows: {len(dover_preview)}/{MAX_VIDEOS}')
else:
    raise RuntimeError('RUN_DOVER must be True for this final bounded run.')

%cd /kaggle/working
print('DOVER_CSV:', DOVER_CSV)


In [ ]:
# 4. Feature extraction with BLIP-base captions on clean metadata subset
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/extract_features.py'),
    '--csv', TRAIN_CSV,
    '--videos', TRAIN_VIDEO_DIR,
    '--out', FEATURES_PATH,
    '--max', str(MAX_VIDEOS),
    '--save-every', '50',
    '--num-frames', '16',
    '--motion-clips', '4',
    '--motion-frames', '16',
    '--caption-frames', '3',
    '--caption-device', CAPTION_DEVICE,
    '--strict-caption',
    '--max-errors-before-abort', '1',
]
if DOVER_CSV:
    cmd += ['--dover-csv', DOVER_CSV]
if not RUN_CAPTION:
    cmd += ['--skip-caption']
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 5. Validate feature schema
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    features = json.load(f)
print('samples:', len(features))
if len(features) != MAX_VIDEOS:
    raise RuntimeError(f'Expected exactly {MAX_VIDEOS} successful samples, got {len(features)}.')

caption_ok = sum(1 for x in features if str(x.get('blip_caption') or x.get('caption') or '').strip())
dover_ok = sum(
    1 for x in features
    if bool((x.get('dover_scores') or {}).get('found', False))
)
print(f'BLIP non-empty captions: {caption_ok}/{len(features)}')
print(f'DOVER matched scores: {dover_ok}/{len(features)}')
if RUN_CAPTION and caption_ok != len(features):
    raise RuntimeError(f'Every sample must have a BLIP caption, got {caption_ok}/{len(features)}.')
if RUN_DOVER and dover_ok != len(features):
    raise RuntimeError(f'Every sample must have a matched DOVER score, got {dover_ok}/{len(features)}.')

first = features[0]
for key in ['clip_frame_embeddings', 'motion_clip_embeddings', 'dover_scores', 'yamnet_embedding_mean', 'metadata_text_embedding', 'caption_embedding', 'rationale_embedding', 'blip_caption']:
    value = first.get(key)
    if isinstance(value, list):
        print(key, 'len:', len(value), 'nested:', len(value[0]) if value and isinstance(value[0], list) else '')
    else:
        print(key, value if key in ['dover_scores', 'blip_caption'] else type(value))
print('First BLIP caption:', (first.get('blip_caption') or first.get('caption') or '')[:300])


In [ ]:
# 6. Train Teacher, Student baseline, and Student+KD on clean metadata subset
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/train.py'),
    '--data', FEATURES_PATH,
    '--save-dir', RESULTS_DIR,
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--teacher-epochs', '40',
    '--student-epochs', '60',
    '--batch', '16',
    '--teacher-hidden', '512',
    '--student-hidden', '256',
    '--teacher-blocks', '2',
    '--selection-metric', 'final_score',
    '--alpha', '0.5',
    '--beta', '0.3',
    '--attn-kd', '0.1',
    '--gamma', '0.2',
    '--delta', '0.2',
]
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 8. Show clean-metadata report and write diagnostics
report_path = os.path.join(RESULTS_DIR, 'final_experiment_report.json')
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)
print(json.dumps(report, indent=2)[:5000])

diag_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/make_diagnostics.py'),
    '--features', FEATURES_PATH,
    '--report', report_path,
    '--subset-csv', TRAIN_CSV,
    '--out-dir', RESULTS_DIR,
    '--bins', str(ECR_BINS),
]
print(' '.join(diag_cmd))
subprocess.run(diag_cmd, check=True)
print('Diagnostics written:')
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, 'diagnostic_*')) + glob.glob(os.path.join(RESULTS_DIR, 'train_metrics.csv')) + glob.glob(os.path.join(RESULTS_DIR, 'feature_diagnostics.csv'))):
    print(path)


In [ ]:
# 9. Package outputs without subset videos
import zipfile
zip_path = f'/kaggle/working/snapugc_lightkd_clean_{REQUIRE_TEXT}_metadata_{MAX_VIDEOS}_outputs.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

package_files = [CLEAN_SOURCE_CSV, SUBSET_CSV, FEATURES_PATH, DOVER_CSV]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in package_files:
        if path and os.path.exists(path):
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))
    for root, _, files in os.walk(RESULTS_DIR):
        for name in files:
            path = os.path.join(root, name)
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))

print(zip_path)
print('Excluded from zip:', SUBSET_VIDEO_DIR)
print('Zip size MB:', round(os.path.getsize(zip_path) / (1024 * 1024), 2))
